<a href="https://colab.research.google.com/github/Umairnu/Data-Science/blob/main/xtanh(2%5Ex).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
import numpy as np
import gc
import time

from tensorflow.keras import layers, Model

print("TensorFlow:", tf.__version__)

# Check GPU
gpus = tf.config.list_physical_devices("GPU")

print("GPUs:", gpus)

if not gpus:
    raise RuntimeError(
        "GPU not detected. Go to Runtime > Change runtime type > GPU."
    )

# Enable memory growth
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print("GPU is ready.")

TensorFlow: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU is ready.


In [2]:
from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy("mixed_float16")

print(
    "Mixed precision policy:",
    mixed_precision.global_policy()
)

Mixed precision policy: <DTypePolicy "mixed_float16">


In [3]:
# ============================================================
# EXPERIMENT SETTINGS
# ============================================================

SEED = 42

BATCH_SIZE = 128

# FIRST TEST:
EPOCHS = 100

# For final experiment change to:
# EPOCHS = 200

INITIAL_LR = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0007

GROWTH_RATE = 32

tf.random.set_seed(SEED)
np.random.seed(SEED)

print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Initial LR:", INITIAL_LR)
print("Momentum:", MOMENTUM)
print("Weight decay:", WEIGHT_DECAY)
print("Growth rate:", GROWTH_RATE)

Batch size: 128
Epochs: 100
Initial LR: 0.1
Momentum: 0.9
Weight decay: 0.0007
Growth rate: 32


In [37]:
# ============================================================
# PROPOSED ACTIVATION
# f(x) = x * tanh(2^x)
# ============================================================

@tf.keras.utils.register_keras_serializable()
class ProposedActivation(tf.keras.layers.Layer):

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, x):
        return x * tf.tanh(tf.pow(tf.cast(2.0, x.dtype), x))

    def get_config(self):
        return super().get_config()

In [8]:
# ============================================================
# LOAD CIFAR-10
# ============================================================

(x_all, y_all), (x_test, y_test) = \
    tf.keras.datasets.cifar10.load_data()

y_all = y_all.reshape(-1)
y_test = y_test.reshape(-1)

print("Full training:", x_all.shape)
print("Test:", x_test.shape)

Full training: (50000, 32, 32, 3)
Test: (10000, 32, 32, 3)


In [9]:
# ============================================================
# DATA SPLIT
# ============================================================

x_train = x_all[:40000]
y_train = y_all[:40000]

x_val = x_all[40000:50000]
y_val = y_all[40000:50000]

# Delete unnecessary full arrays
del x_all
del y_all

gc.collect()

print("Training:", x_train.shape)
print("Validation:", x_val.shape)
print("Test:", x_test.shape)

Training: (40000, 32, 32, 3)
Validation: (10000, 32, 32, 3)
Test: (10000, 32, 32, 3)


In [10]:
# ============================================================
# NORMALIZATION
# ============================================================

mean = np.array(
    [0.4914, 0.4822, 0.4465],
    dtype=np.float32
)

std = np.array(
    [0.2470, 0.2435, 0.2616],
    dtype=np.float32
)


def normalize(x):

    x = x.astype(np.float32) / 255.0

    x = (x - mean) / std

    return x


x_train = normalize(x_train)
x_val = normalize(x_val)
x_test = normalize(x_test)

gc.collect()

print("Normalization complete.")

Normalization complete.


In [11]:
# ============================================================
# DATA AUGMENTATION
# ============================================================

def augment(image, label):

    # 4-pixel padding
    image = tf.pad(
        image,
        [
            [4, 4],
            [4, 4],
            [0, 0]
        ],
        mode="REFLECT"
    )

    # Random 32x32 crop
    image = tf.image.random_crop(
        image,
        size=[32, 32, 3]
    )

    # Random horizontal flip
    image = tf.image.random_flip_left_right(
        image
    )

    return image, label

In [13]:
# ============================================================
# TRAIN DATASET
# ============================================================

train_dataset = tf.data.Dataset.from_tensor_slices(
    (x_train, y_train)
)

train_dataset = (
    train_dataset
    .shuffle(
        buffer_size=40000,
        seed=SEED,
        reshuffle_each_iteration=True
    )
    .map(
        augment,
        num_parallel_calls=tf.data.AUTOTUNE
    )
    .batch(
        BATCH_SIZE,
        drop_remainder=True
    )
    .prefetch(
        tf.data.AUTOTUNE
    )
)


# ============================================================
# VALIDATION DATASET
# ============================================================

val_dataset = tf.data.Dataset.from_tensor_slices(
    (x_val, y_val)
)

val_dataset = (
    val_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)


# ============================================================
# TEST DATASET
# ============================================================

test_dataset = tf.data.Dataset.from_tensor_slices(
    (x_test, y_test)
)

test_dataset = (
    test_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)


print("Datasets created.")
del x_train
del y_train
del x_val
del y_val
del x_test
del y_test

gc.collect()

print("Unused arrays released.")

Datasets created.
Unused arrays released.


In [14]:
# ============================================================
# CONVOLUTION
# ============================================================

def conv_layer(filters, kernel_size):

    return layers.Conv2D(
        filters=filters,
        kernel_size=kernel_size,
        padding="same",
        use_bias=False,
        kernel_initializer=(
            tf.keras.initializers.GlorotUniform()
        )
    )

In [38]:
# ============================================================
# DENSE LAYER
# ============================================================

@tf.keras.utils.register_keras_serializable()
class DenseLayer(tf.keras.layers.Layer):

    def __init__(self, growth_rate, **kwargs):
        super().__init__(**kwargs)

        self.growth_rate = growth_rate

        self.bn1 = layers.BatchNormalization()
        self.act1 = ProposedActivation()

        self.conv1 = layers.Conv2D(
            4 * growth_rate,
            kernel_size=1,
            padding="same",
            use_bias=False
        )

        self.bn2 = layers.BatchNormalization()
        self.act2 = ProposedActivation()

        self.conv2 = layers.Conv2D(
            growth_rate,
            kernel_size=3,
            padding="same",
            use_bias=False
        )

    def call(self, x, training=None):

        y = self.bn1(x, training=training)
        y = self.act1(y)
        y = self.conv1(y)

        y = self.bn2(y, training=training)
        y = self.act2(y)
        y = self.conv2(y)

        return tf.concat([x, y], axis=-1)

    def get_config(self):

        config = super().get_config()

        config.update({
            "growth_rate": self.growth_rate
        })

        return config

In [1]:
# ============================================================
# DENSE BLOCK
# ============================================================

@tf.keras.utils.register_keras_serializable()
class DenseBlock(tf.keras.layers.Layer):

    def __init__(
        self,
        num_layers,
        growth_rate,
        **kwargs
    ):
        super().__init__(**kwargs)

        self.num_layers = num_layers
        self.growth_rate = growth_rate

        self.dense_layers = [
            DenseLayer(growth_rate)
            for _ in range(num_layers)
        ]

    def call(self, x, training=None):

        for dense_layer in self.dense_layers:
            x = dense_layer(
                x,
                training=training
            )

        return x

    def get_config(self):

        config = super().get_config()

        config.update({
            "num_layers": self.num_layers,
            "growth_rate": self.growth_rate
        })

        return config

NameError: name 'tf' is not defined

In [17]:
# ============================================================
# TRANSITION BLOCK
# ============================================================

class TransitionBlock(layers.Layer):

    def __init__(self, filters):

        super().__init__()

        self.bn = layers.BatchNormalization()

        self.act = ProposedActivation()

        self.conv = conv_layer(
            filters,
            1
        )

        self.pool = layers.AveragePooling2D(
            pool_size=2,
            strides=2
        )


    def call(self, x, training=False):

        x = self.bn(
            x,
            training=training
        )

        x = self.act(x)

        x = self.conv(x)

        x = self.pool(x)

        return x

In [18]:
# ============================================================
# DENSENET-121 FOR CIFAR-10
# ============================================================

def build_densenet121_cifar10():

    inputs = layers.Input(
        shape=(32, 32, 3)
    )


    # ========================================================
    # Initial convolution
    # ========================================================

    x = conv_layer(
        64,
        3
    )(inputs)


    # ========================================================
    # DENSE BLOCK 1
    # 6 layers
    # ========================================================

    x = DenseBlock(
        num_layers=6,
        growth_rate=GROWTH_RATE
    )(x)

    filters = x.shape[-1]

    x = TransitionBlock(
        filters // 2
    )(x)


    # ========================================================
    # DENSE BLOCK 2
    # 12 layers
    # ========================================================

    x = DenseBlock(
        num_layers=12,
        growth_rate=GROWTH_RATE
    )(x)

    filters = x.shape[-1]

    x = TransitionBlock(
        filters // 2
    )(x)


    # ========================================================
    # DENSE BLOCK 3
    # 24 layers
    # ========================================================

    x = DenseBlock(
        num_layers=24,
        growth_rate=GROWTH_RATE
    )(x)

    filters = x.shape[-1]

    x = TransitionBlock(
        filters // 2
    )(x)


    # ========================================================
    # DENSE BLOCK 4
    # 16 layers
    # ========================================================

    x = DenseBlock(
        num_layers=16,
        growth_rate=GROWTH_RATE
    )(x)


    # ========================================================
    # FINAL BN + PROPOSED ACTIVATION
    # ========================================================

    x = layers.BatchNormalization()(x)

    x = ProposedActivation()(x)


    # ========================================================
    # GLOBAL AVERAGE POOLING
    # ========================================================

    x = layers.GlobalAveragePooling2D()(x)


    # ========================================================
    # CIFAR-10 CLASSIFIER
    #
    # dtype=float32 is important with mixed precision
    # ========================================================

    outputs = layers.Dense(
        10,
        activation="softmax",
        dtype="float32",
        kernel_initializer=(
            tf.keras.initializers.GlorotUniform()
        )
    )(x)


    return Model(
        inputs=inputs,
        outputs=outputs
    )

In [19]:
# ============================================================
# BUILD MODEL
# ============================================================

model = build_densenet121_cifar10()

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 32, 32, 64)     │         1,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_block (DenseBlock)        │ (None, 32, 32, 256)    │       338,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transition_block                │ (None, 16, 16, 128)    │        33,792 │
│ (TransitionBlock)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_block_1 (DenseBlock)      │ (None, 16, 16, 512)    │       930,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transition_block_1              │ (None, 8, 8, 256)      │       133,120 │
│ (TransitionBlock)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_block_2 (DenseBlock)      │ (None, 8, 8, 1024)     │     2,873,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transition_block_2              │ (None, 4, 4, 512)      │       528,384 │
│ (TransitionBlock)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_block_3 (DenseBlock)      │ (None, 4, 4, 1024)     │     2,186,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_119         │ (None, 4, 4, 1024)     │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ proposed_activation_119         │ (None, 4, 4, 1024)     │             0 │
│ (ProposedActivation)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        10,250 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,039,818 (26.85 MB)

 Trainable params: 6,956,298 (26.54 MB)

 Non-trainable params: 83,520 (326.25 KB)

In [20]:
# ============================================================
# QUICK FORWARD PASS
# ============================================================

for images, labels in train_dataset.take(1):

    predictions = model(images)

    print("Input :", images.shape)
    print("Output:", predictions.shape)
    print("Labels:", labels.shape)

Input : (128, 32, 32, 3)
Output: (128, 10)
Labels: (128,)


In [21]:
# ============================================================
# SGD + MOMENTUM
# ============================================================

optimizer = tf.keras.optimizers.SGD(
    learning_rate=INITIAL_LR,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY
)

In [22]:
# ============================================================
# COMPILE
# ============================================================

model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [25]:
# ============================================================
# LEARNING RATE SCHEDULE
# ============================================================

def lr_schedule(epoch, lr):

    if epoch > 0 and epoch % 60 == 0:
        lr = lr * 0.2

    return lr


lr_callback = tf.keras.callbacks.LearningRateScheduler(
    lr_schedule,
    verbose=1
)

In [26]:
# ============================================================
# MODEL CHECKPOINT
# ============================================================

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "best_proposed_activation.keras",
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    verbose=1
)

In [27]:
# ============================================================
# TRAIN
# ============================================================

start_time = time.time()

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=[
        lr_callback,
        checkpoint
    ]
)

training_time = time.time() - start_time

print(
    f"\nTraining time: "
    f"{training_time / 60:.2f} minutes"
)


Epoch 1: LearningRateScheduler setting learning rate to 0.10000000149011612.
Epoch 1/100
312/312 ━━━━━━━━━━━━━━━━━━━━ 0s 187ms/step - accuracy: 0.3324 - loss: 1.9620
Epoch 1: val_accuracy improved from None to 0.49800, saving model to best_proposed_activation.keras

Epoch 1: finished saving model to best_proposed_activation.keras
312/312 ━━━━━━━━━━━━━━━━━━━━ 327s 303ms/step - accuracy: 0.4196 - loss: 1.6490 - val_accuracy: 0.4980 - val_loss: 1.3906 - learning_rate: 0.1000

Epoch 2: LearningRateScheduler setting learning rate to 0.10000000149011612.
Epoch 2/100
312/312 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step - accuracy: 0.5762 - loss: 1.2025
Epoch 2: val_accuracy improved from 0.49800 to 0.61730, saving model to best_proposed_activation.keras

Epoch 2: finished saving model to best_proposed_activation.keras
312/312 ━━━━━━━━━━━━━━━━━━━━ 65s 209ms/step - accuracy: 0.6051 - loss: 1.1155 - val_accuracy: 0.6173 - val_loss: 1.1148 - learning_rate: 0.1000

Epoch 3: LearningRateScheduler setting le

In [36]:
best_model = tf.keras.models.load_model(
    "best_proposed_activation.keras",
    custom_objects={
        "DenseBlock": DenseBlock,
        "TransitionBlock": TransitionBlock,
        "ProposedActivation": ProposedActivation
    },
    compile=False
)

TypeError: <class 'keras.src.models.functional.Functional'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras.src.models.functional', 'class_name': 'Functional', 'config': {}, 'registered_name': 'Functional', 'build_config': {'input_shape': None}, 'compile_config': None}.

Exception encountered: <class '__main__.DenseBlock'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': None, 'class_name': 'DenseBlock', 'config': {'num_layers': 6, 'growth_rate': 32, 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'mixed_float16'}, 'registered_name': None, 'shared_object_id': 132240